In [ ]:
# 优化器 optimizer
# 前面用的都是SGD（stochastic gradient descent），他是在整个权重空间执行下降
# 有几个问题没有解决：
# （1）对于非凸优化问题容易陷入局部最优解法，没法跨过鞍点，收敛的慢
# （2）没有自适应学习率，整个梯度下降的过程中都采取一样的学习率，容易在驻点处左右横跳
# （3）所有参数的学习率都一样，没有针对每个参数独有一份参考值
import numpy as np
# 首先来看SGD随机梯度下降
# 只是前面的内容抽象成了一个类，没有涉及任何的参数存储啥的，就是一个计算式子；下降就完事儿
class SGD():
    def __init__(self,lr = 0.01):
        self.lr = lr

    def update(self,params,grads):# 整个网络的参数和整个网络的梯度
        # 可变对象引用传递，原地修改
        for param in params.keys():
            params[param] = params[param] - self.lr*grads[param]
    

# sgd的优缺点很明显，优点是简单，易实现，不用存储额外的参数，但是缺点就是文件最开始提到的内容，
# 最容易想到的就是我给学习率上个刹车/油门，根据梯度的不同我做一个累加，梯度越“快”（相同方向上累加的越多）更新越大
# 反之如果梯度在震荡，那就是离“最优值”不远咯，要更新的慢一点，这是一种很朴素的想法，也很实用
# 这个刹车和油门称为动量（Momentum），下面的方法就是sgd加了个动量

class SGDMomentum():
    def __init__(self,alpha=0.99,lr=0.1):
        # 动量有两个可调超参数，alpha是动量前的系数，也就是刹车油门的衰减系数
        self.alpha = alpha
        self.lr = lr
        # 存储的是所有参数历史的动量，这是必要手段，从定义就能知道必须要存
        # 当然这也会引起额外的存储，约等于将整个网络所需要的存储翻倍
        self.momentum = None

    def update(self,params,grads):
        # momentum = alpha * momentum - lr * grads
        # w = w+momentum
        if self.momentum == None:
            # 相当于初始化为0
            self.momentum = {}
            for key in params.keys():
                self.momentum[key] = np.zeros_like(params[key])
        
        for key in params.keys():
            self.momentum[key] = self.alpha*self.momentum[key] - self.lr*grads[key]
            params[key] += self.momentum[key]

# 动量很明显的第一个缺点就是累积历史的动量可能会在“最优点”处冲过头，导致来来回回
# 第二个缺点就是lr和alpha强耦合，不容易调参；当lr和alpha都很大的时候都有可能不收敛
# 从第一个缺点可以导出，不适合非平稳目标，比如像GAN网络、在线学习这种梯度方向频繁震荡的时候，只用历史累积梯度可能会帮倒忙，永远不能收敛
# 最后，仅用动量相当于没有学习率衰减，学习率还是从头到尾不变，这明显不合理。针对这个问题，引出了AdaGrad。

class AdaGrad():
    def __init__(self,lr=0.1):
        # 跟动量一样，要存一个h
        self.lr = lr
        self.h = None

    def update(self,params,grads):
        # h = h + grads*grads 哈达玛积
        # w = w - 1/sqrt(h) * grads
        if self.h == None:
            for key in params.keys():
                self.h = {}
                self.h[key] = np.zeros_like(params[key])

        for key in params.keys():
            self.h[key] = self.h[key] + grads[key]*grads[key]
            # 1e-7防止除0
            params[key] -= self.lr * grads[key] /np.sqrt(self.h[key]+ 1e-7) 

# Adagrad针对每个参数都设计一个衰减（learning rate decay），但由于是平方和相加，h是一个单调递增函数
# 体现到第二个式子就是梯度在不断衰减，趋向于零。这个过程不可逆，后期模型可能出现学不动的情况。
# 适合稀疏特征（稀有特征梯度累积慢 → 学习率大 → 每次出现都能充分更新）、更新步数小的场景。
# 并且对初始lr敏感，初始lr大了再叠加单调递增函数很快就直接学不动了，发散了

# RMSProp在AdaGrad的基础上相当于加了一个窗口值，不要过分关注历史累积的梯度平方的值，多关注当前的梯度
# 具体做法就是在计算h的时候给当前梯度和之前的h值各给一个权重，这样不仅保证了以前的h值快速衰减，还能保证当前梯度的有效性
class RMSProp():
    def __init__(self,lr,alpha=0.9):
        # alpha就是当前梯度和之前的h的权重
        self.lr = lr
        self.alpha = alpha
        self.h = None

    def update(self,params,grads):
        # h = alpha*h+(1-alpha)*grad*grad
        # w = w-lr*grad/sqrt(h)
        if self.h == None:
            for key in params.keys():
                self.h = {}
                self.h[key] = np.zeros_like(params[key])

        for key in params.keys():
            self.h[key] = self.alpha*self.h[key]+(1-self.alpha)*grads[key]*grads[key]
            params[key] -= self.lr*grads[key]/sqrt(self.h[key]+1e-7)

# RMSProp从AdaGrad发展而来，因此优点有：自适应学习率、学习率不会衰减到0，只看近处梯度适应快速变化等
# 缺点为没有动量可能会陷在“长且窄”的峡谷里面、初始由于h为0，sqrt(alpha*g0^2)值可能会偏大，导致开始学习的时候震荡

# Adam就融合了动量+RMSProp的思想，让优化器优化的函数能冲过峡谷。
# Adam中使用的是一阶矩和二阶矩，无偏估计，其本质是用于在训练开始时还原正常的学习率
# 可以设前5轮的grad=10，β1=β2=0.9 手动计算一边，会发现如果不用还原的话动量和v都会偏小
# 作用到分子分母就是动量*lr整体偏小，1/sqrt(v)整体偏大，所以需要在第一步修复步长，而不是像rmsprop一样过了很久才收敛到正常的学习率
class Adam():
    def __init__(self,lr,beta1,beta2,weight_decay=0.01):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        # 如果这里要weight decay的话，需要在计算梯度的地方加上正则化项的导数
        # grad = grad + weight_decay*param
        self.weight_decay = weight_decay
        # 计算当前迭代的轮数
        self.t = 1

        # 存储一阶矩和二阶矩
        # 所以adam和adamw需要参数额外2倍的存储（不计算混合精度训练）
        self.m = None
        self.v = None

    def update(self,params,grad):
        # mt = beta1*mt-1 + (1-beta1)*gt-1
        # mt' = mt/1-beta1^t
        # vt = beta2*vt-1 + (1-beta2)*gt-1^2
        # vt' = vt/1-beta2^t
        # w = w - lr*mt'/sqrt(vt'+1e-4)
        if self.m == None:
            self.m = {}
            self.v = {}
            for key in params.keys():
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])

        for key in params.keys():
            # L2正则
            decay_grad = grad[key] + self.weight_decay*params[key]

            self.m = self.beta1*self.m+(1-self.beta1)*decay_grad
            mt_correction = self.m/(1-self.beta1)**self.t
            self.v = self.beta2*self.v+(1-self.beta2)*decay_grad*decay_grad
            vt_correction = self.v/(1-self.beta2)**self.t
            params[key] -= self.lr*mt_correction /np.sqrt(vt_correction + 1e-4)
        self.t += 1

# 对于adam的改进就是一行，将正则项拿出来了，让其和梯度解耦
# 原本的L2正则化项直接加载损失函数上，L = L0+1/2*α*||θ||
# 但这相当于在加在一阶矩上后要除以二阶矩，导致正则化力度偏弱
# 正则化用于权重衰减（weight decay），把权重向0处拉，对大权重做惩罚
class AdamW():
    def __init__(self,lr,beta1,beta2,weight_decay):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.t = 1

        self.m = None
        self.v = None


    def update(self,params,grads):
        if self.m == None:
            for key in params.keys():
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
        
        for key in params.keys():
            self.m[key] = self.beta1*self.m[key] + (1-self.beta1)*grads[key]
            mt_correction = self.m[key]/(1-self.beta1)**self.t
            self.v[key] = self.beta2*self.v[key] + (1-self.beta2)*grads[key]*grads[key]
            vt_correction = self.v[key]/(1-self.beta2)**self.t
            # L2正则在这里解耦
            params[key] = params[key] - self.lr*(self.mt_correction/(np.sqrt(vt_correction)+1e-7)+self.weight_decay*params[key])
        self.t += 1